# 📈 Backtest Report – Quant Analyst
**Workflow**: Signal Prep → Vectorized Backtest → Monte Carlo Simulation → Monthly PnL Analytics
---

In [1]:
from qtrader.output.analyst import AnalystSession, RoleContext
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

session = AnalystSession(role=RoleContext.ANALYST)
SYMBOL = "ETH-USD"
TIMEFRAME = "1d"
TRANSACTION_COST = 0.0005
INITIAL_CAPITAL = 100000

PLOTLY_DARK = dict(template="plotly_dark", plot_bgcolor='#0f1117', paper_bgcolor='#0f1117')

## 1. Strategy Definition
Generating internal signals for backtesting.

In [2]:
try:
    df = session.load_ohlcv(SYMBOL, TIMEFRAME, days=730)
except Exception:
    df = session.sample_ohlcv(symbol="BTC", days=365)

df = session.make_returns(df)
df = session.add_rolling_features(df, windows=[5, 20])

# Basic Trend-Following Cross: SMA(21) vs SMA(50)
df = df.with_columns(
    pl.when(pl.col('sma_5') > pl.col('sma_20')).then(1).otherwise(-1).alias('signal')
)
df.head()

timestamp,open,high,low,close,volume,symbol,tf,returns,sma_5,vol_5,sma_20,vol_20,avg_gain,avg_loss,rsi_14,signal
"datetime[μs, Asia/Ho_Chi_Minh]",f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,i32
2026-03-10 07:00:00 +07,1993.18,2088.83,1991.09,2036.8,175603.576719,"""ETH-USD""","""1d""",null,null,null,null,null,null,null,null,-1
2026-03-11 07:00:00 +07,2037.0,2086.49,2007.0,2052.25,166606.191789,"""ETH-USD""","""1d""",0.007585,null,null,null,null,null,null,null,-1
2026-03-12 07:00:00 +07,2052.42,2096.09,2018.15,2074.0,154623.308753,"""ETH-USD""","""1d""",0.010598,null,null,null,null,null,null,null,-1
2026-03-13 07:00:00 +07,2073.78,2211.62,2070.09,2092.81,224337.395247,"""ETH-USD""","""1d""",0.009069,null,null,null,null,null,null,null,-1
2026-03-14 07:00:00 +07,2092.81,2106.06,2061.91,2096.7,50376.203744,"""ETH-USD""","""1d""",0.001859,2070.512,25.81268,null,null,null,null,null,-1


## 2. Interactive Performance Overview
Strategy Equity Curve vs Buy & Hold Benchmark.

In [3]:
pf = session.run_vbt_backtest(df, signal_col='signal', transaction_cost=TRANSACTION_COST)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['timestamp'], y=pl.Series("equity_curve", pf.value()), name='Strategy', line=dict(color='#34d399')))
bh = (1 + df['returns'].fill_null(0)).cum_prod()
fig.add_trace(go.Scatter(x=df['timestamp'], y=bh, name='Buy & Hold', line=dict(color='#64748b', dash='dash')))

fig.update_layout(
    title=f"{SYMBOL} Strategy Cumulative Performance",
    yaxis_title="Relative Equity",
    **PLOTLY_DARK
)
fig.show()

## 3. Monte Carlo Robustness Test
Shuffling daily returns 250 times to test if the strategy's performance is luck-based.

In [4]:
n_sims = 250
strat_returns = pl.Series("net_return", pf.returns()).drop_nulls().to_numpy()
sim_results = []

for _ in range(n_sims):
    shuffled = np.random.choice(strat_returns, size=len(strat_returns), replace=True)
    sim_results.append(np.cumprod(1 + shuffled))

fig_mc = go.Figure()
for sim in sim_results[:50]: # plot subset for speed
    fig_mc.add_trace(go.Scatter(y=sim, mode='lines', line=dict(width=0.5), opacity=0.3, showlegend=False, line_color='#94a3b8'))
    
fig_mc.add_trace(go.Scatter(y=np.percentile(sim_results, 50, axis=0), name='Median Sim', line=dict(color='#fef08a', width=2)))
fig_mc.add_trace(go.Scatter(y=np.cumprod(1 + strat_returns), name='Actual Strategy', line=dict(color='#34d399', width=3)))

fig_mc.update_layout(
    title=f"Monte Carlo Simulation ({n_sims} paths)",
    xaxis_title="Days",
    yaxis_title="Equity",
    **PLOTLY_DARK
)
fig_mc.show()

## 4. Advanced Risk Metrics
Computing drawdown profiles and VaR.

In [5]:
metrics = session.compute_extended_metrics(pl.Series("equity_curve", pf.value()))
metrics_df = pl.DataFrame({"Metric": list(metrics.keys()), "Value": list(metrics.values())})
display(metrics_df)

equity = pl.Series("equity_curve", pf.value()).to_numpy()
peaks = np.maximum.accumulate(equity)
drawdowns = (equity - peaks) / peaks

fig_dd = px.area(x=df['timestamp'], y=drawdowns, title="Drawdown Profile", labels={'y': 'Drawdown %'})
fig_dd.update_traces(fillcolor='#f87171', line_color='#ef4444')
fig_dd.update_layout(**PLOTLY_DARK)
fig_dd.show()

Metric,Value
str,f64
"""total_return""",0.0
"""annualized_return""",0.0
"""annualized_vol""",0.0
"""sharpe_ratio""",0.0
"""max_drawdown""",0.0
…,…
"""calmar_ratio""",0.0
"""win_rate""",0.0
"""profit_factor""",inf


## 5. Monthly Return Heatmap
Visualizing seasonality and periodicity in returns.

In [6]:
# Robust monthly heatmap using AnalystSession helper
monthly_ret = session.get_monthly_returns(bt)
heatmap_data = monthly_ret.select(pl.all().exclude(['year', 'YTD']))

fig_heat = px.imshow(
    heatmap_data.to_numpy(),
    x=heatmap_data.columns,         # Jan...Dec
    y=monthly_ret['year'].to_numpy(),
    color_continuous_scale='RdYlGn',
    origin='upper',
    title="Monthly Return Heatmap (%)"
)
fig_heat.update_layout(**PLOTLY_DARK)
fig_heat.show()

NameError: name 'bt' is not defined